Siphon, a library used to access THREDDS, a data server that provides web-based access to scientific datasets and allows specific data extraction. (So you don't have to download everything needlessley)

In [3]:
%pip install siphon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 kB 4.8 MB/s eta 0:00:00


Here we get and list available forecast files from 2015-01-15, The whole dataset for GFS 0.25 spans from 2015-01-15 to present

In [4]:
from siphon.catalog import TDSCatalog
catalog = TDSCatalog('https://thredds.rda.ucar.edu/thredds/catalog/files/g/d084001/2015/20150115/catalog.xml')
print(catalog.datasets)

['gfs.0p25.2015011500.f000.grib2', 'gfs.0p25.2015011500.f003.grib2', 'gfs.0p25.2015011500.f006.grib2', 'gfs.0p25.2015011500.f009.grib2', 'gfs.0p25.2015011500.f012.grib2', 'gfs.0p25.2015011500.f015.grib2', 'gfs.0p25.2015011500.f018.grib2', 'gfs.0p25.2015011500.f021.grib2', 'gfs.0p25.2015011500.f024.grib2', 'gfs.0p25.2015011500.f027.grib2', 'gfs.0p25.2015011500.f030.grib2', 'gfs.0p25.2015011500.f033.grib2', 'gfs.0p25.2015011500.f036.grib2', 'gfs.0p25.2015011500.f039.grib2', 'gfs.0p25.2015011500.f042.grib2', 'gfs.0p25.2015011500.f045.grib2', 'gfs.0p25.2015011500.f048.grib2', 'gfs.0p25.2015011500.f051.grib2', 'gfs.0p25.2015011500.f054.grib2', 'gfs.0p25.2015011500.f057.grib2', 'gfs.0p25.2015011500.f060.grib2', 'gfs.0p25.2015011500.f063.grib2', 'gfs.0p25.2015011500.f066.grib2', 'gfs.0p25.2015011500.f069.grib2', 'gfs.0p25.2015011500.f072.grib2', 'gfs.0p25.2015011500.f075.grib2', 'gfs.0p25.2015011500.f078.grib2', 'gfs.0p25.2015011500.f081.grib2', 'gfs.0p25.2015011500.f084.grib2', 'gfs.0p25.201

**Understanding the file naming convention:**

`gfs.0p25.YYYYMMDDHH.fXXX.grib2`

Breaking It Down:

`gfs.0p25` → This refers to the GFS(Global Forecast System) model with 0.25-degree resolution (high resolution).

`YYYYMMDDHH` → The initialization time of the forecast in UTC. (REAL OBSERVATION STARTING POINT)

2015011500 → January 15, 2015, at 00:00 UTC. (02:00 SWEDISH SUMMER TIME)

2015011506 → January 15, 2015, at 06:00 UTC. (08:00 SWEDISH SUMMER TIME)

2015011512 → January 15, 2015, at 12:00 UTC. (14:00 SWEDISH SUMMER TIME)

2015011518 → January 15, 2015, at 18:00 UTC. (20:00 SWEDISH SUMMER TIME)

`fXXX` → The forecast hour.

f000 → This is the analysis time (the actual observations at initialization time).

f003 → 3-hour forecast from initialization.

f006 → 6-hour forecast from initialization.

f009 → 9-hour forecast from initialization.

…and so on, in 3-hour increments.

So, Let's extract the datasets spanning a 10 days prognosis, Meaning 240 hours onwards and start at 12UTC initialization.

In [5]:
# List all datasets
all_datasets = list(catalog.datasets.keys())

init_time = "2015011512"  # (12 UTC) init
prognosis = 168 #forecast hours <= 240 (interval of 3 hours) max is 384 (16 days)
# Filter datasets that have "2015011512" (12 UTC) and forecast hours <= 240
filtered_files = [f for f in all_datasets if init_time in f and int(f.split(".f")[-1][:3]) <= prognosis]

# Print the selected files
print("Selected files for processing:", filtered_files)

Selected files for processing: ['gfs.0p25.2015011512.f000.grib2', 'gfs.0p25.2015011512.f003.grib2', 'gfs.0p25.2015011512.f006.grib2', 'gfs.0p25.2015011512.f009.grib2', 'gfs.0p25.2015011512.f012.grib2', 'gfs.0p25.2015011512.f015.grib2', 'gfs.0p25.2015011512.f018.grib2', 'gfs.0p25.2015011512.f021.grib2', 'gfs.0p25.2015011512.f024.grib2', 'gfs.0p25.2015011512.f027.grib2', 'gfs.0p25.2015011512.f030.grib2', 'gfs.0p25.2015011512.f033.grib2', 'gfs.0p25.2015011512.f036.grib2', 'gfs.0p25.2015011512.f039.grib2', 'gfs.0p25.2015011512.f042.grib2', 'gfs.0p25.2015011512.f045.grib2', 'gfs.0p25.2015011512.f048.grib2', 'gfs.0p25.2015011512.f051.grib2', 'gfs.0p25.2015011512.f054.grib2', 'gfs.0p25.2015011512.f057.grib2', 'gfs.0p25.2015011512.f060.grib2', 'gfs.0p25.2015011512.f063.grib2', 'gfs.0p25.2015011512.f066.grib2', 'gfs.0p25.2015011512.f069.grib2', 'gfs.0p25.2015011512.f072.grib2', 'gfs.0p25.2015011512.f075.grib2', 'gfs.0p25.2015011512.f078.grib2', 'gfs.0p25.2015011512.f081.grib2', 'gfs.0p25.201501

siphon and THREDDS allows us to query for subset regions, So let's define a subset region for Sweden

In [6]:
# Define Sweden's bounding box
lat_min, lat_max = 55.0, 70.0
lon_min, lon_max = 10.0, 25.0

Now we need to download each subset for each file separately and then combine each dataset alongside time coord

Temperature_surface

Wind_speed_gust_surface

Relative_humidity_sigma

Relative_humidity_height_above_ground

Volumetric_Soil_Moisture_Content_depth_below_surface_layer

Soil_temperature_depth_below_surface_layer

Cloud_water_entire_atmosphere_single_layer

Precipitable_water_entire_atmosphere_single_layer



In [18]:
from datetime import datetime
import xarray as xr
import pandas as pd
from io import BytesIO
import time

ds_list = []

#graderingstilfallen
#graderingsdatum
#

for file in filtered_files:
    dataset = catalog.datasets[file]

    # Check if NCSS is available
    if 'NetcdfSubset' in dataset.access_urls:

        ncss = dataset.subset()
        #print("Available variables:", ncss.variables)

        query = ncss.query()
        # Subset query for Sweden and the Temperature_surface variable
        query.lonlat_box(north=lat_max, south=lat_min, east=lon_max, west=lon_min)
        try:
          try:
            query.variables('Total_precipitation_surface_3_Hour_Accumulation')
            query.accept('netcdf4')


            print(f"Fetching data from {file}...")

            #First doesnt have any TOtal_precipitation_surface_3_hour_acccum how to skip?
            data = ncss.get_data(query)
            ds = xr.open_dataset(BytesIO(data))
          # Some has time1 instead of time? no idea why
            if 'time1' in ds.coords:
                ds = ds.rename({'time1': 'time'})

            ds_list.append(ds)
          except Exception as e:
            #print(e)
            pass
        except:
          pass



      #print(ds.variables)
      #time.sleep(99999)



# Now, concatenate all datasets along the time dimension.
if ds_list:
    ds_combined = xr.concat(ds_list, dim="time")
    print("Final dataset structure:")
    print(ds_combined)
else:
    print("No data was retrieved. Please check dataset availability.")

print(ds_combined.variables)


# Print a summary of the dataset
# print(ds)

# List all coordinate names
#print("Coordinates:", list(ds.coords))

# List all variable names
# print("Variables:", list(ds.variables))

#print(ds["reftime"].item())
#time.sleep(999999)

Fetching data from gfs.0p25.2015011512.f000.grib2...
Fetching data from gfs.0p25.2015011512.f003.grib2...
Fetching data from gfs.0p25.2015011512.f006.grib2...
Fetching data from gfs.0p25.2015011512.f009.grib2...
Fetching data from gfs.0p25.2015011512.f012.grib2...
Fetching data from gfs.0p25.2015011512.f015.grib2...
Fetching data from gfs.0p25.2015011512.f018.grib2...
Fetching data from gfs.0p25.2015011512.f021.grib2...
Fetching data from gfs.0p25.2015011512.f024.grib2...
Fetching data from gfs.0p25.2015011512.f027.grib2...
Fetching data from gfs.0p25.2015011512.f030.grib2...
Fetching data from gfs.0p25.2015011512.f033.grib2...
Fetching data from gfs.0p25.2015011512.f036.grib2...
Fetching data from gfs.0p25.2015011512.f039.grib2...
Fetching data from gfs.0p25.2015011512.f042.grib2...
Fetching data from gfs.0p25.2015011512.f045.grib2...
Fetching data from gfs.0p25.2015011512.f048.grib2...
Fetching data from gfs.0p25.2015011512.f051.grib2...
Fetching data from gfs.0p25.2015011512.f054.gr

In [20]:
import numpy as np
# Replace NaT values in time_bounds with corresponding values from time1_bounds
ds_combined['time_bounds'] = ds_combined['time_bounds'].where(
    ~np.isnat(ds_combined['time_bounds']),
    ds_combined['time1_bounds']
)

# Optionally, drop time1_bounds if no longer needed
ds_combined = ds_combined.drop_vars('time1_bounds')

output_filename = "gfs_sweden_regn.nc"
ds_combined.to_netcdf(output_filename)
print(f"Saved dataset to {output_filename}")

Saved dataset to gfs_sweden_regn.nc


Input target lat&lon and get back JSON file with values

In [22]:
import pandas as pd
import json
# Define your target latitude and longitude
#öland
target_lat = 56.7  # replace with your specific latitude
target_lon = 16.6  # replace with your specific longitude

#select the nearest grid point using .sel()
point_data = ds_combined.sel(latitude=target_lat, longitude=target_lon, method="nearest")

#extract the time and precipitation arrays
time_values = point_data["time"].values
precip_values = point_data["Total_precipitation_surface_3_Hour_Accumulation"].values

time_precip_list = []
for t, precip in zip(time_values, precip_values):
    #Convert numpy.datetime64 to a formatted string
    time_str = pd.to_datetime(t).strftime("%Y-%m-%dT%H:%M:%SZ")
    time_precip_list.append({"time": time_str, "precipitation": float(precip)})

#Convert the list to a JSON string
json_output = json.dumps(time_precip_list, indent=2)
print(json_output)

[
  {
    "time": "2015-01-15T13:30:00Z",
    "precipitation": 0.30000001192092896
  },
  {
    "time": "2015-01-15T19:30:00Z",
    "precipitation": 1.7999999523162842
  },
  {
    "time": "2015-01-16T01:30:00Z",
    "precipitation": 0.30000001192092896
  },
  {
    "time": "2015-01-16T07:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-16T13:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-16T19:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-17T01:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-17T07:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-17T13:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-17T19:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-18T01:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-18T07:30:00Z",
    "precipitation": 0.699999988079071
  },
  {
    "time": "2015-01-18T13:30:00Z",
    "precipitation": 0.0
  },
  {
    "time": "2015-01-18